# Cross-model tournament — 260914 vs 260723, finished policies

Runs central models from both batches against each other in inference and writes one
behavioural-log JSON per matchup (the format `octagon_analysis` consumes).

**The question.** 260914 central models choose the High wall on every trial (`p_high` = 1.00) where
260723 central models choose it about 80% of the time. Does that stronger High preference actually
earn more reward per unit time **in competition**?

**Every entrant is at its own batch's finished policy**: 260914 at ~7.00e6 steps, 260723 at ~4.00e6.
The two are not step-matched, and deliberately so. 260723 converged by about 3e6 (its episode length
levelled off while the learning rate was still near half peak) and its linear schedule then annealed
it to a stop, so it cannot be improved by spending more steps within its own recipe. 260914 uses a
constant learning rate and is still improving at 7e6. Matching step counts at 4e6 would therefore
compare a converged, annealed model against one running mid-flight at eight times the learning rate,
which matches the step count without matching the training state. See
`agent_training/training_convergence_260723_vs_260914.ipynb`.

| entrant | batch | step | p_high | onset centrality | score_rate | path_ratio | role |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `260914_0020` | 260914 | 7,000,371 | 1.000 | 0.870 | 0.488 | 1.151 | central, most typical of its batch |
| `260914_0014` | 260914 | 7,000,221 | 1.000 | 0.880 | 0.473 | 1.159 | central, 2nd most typical |
| `260723_0034` | 260723 | 4,000,042 | 0.838 | 0.880 | 0.444 | 1.129 | central, most typical of its batch |
| `260723_0039` | 260723 | 4,000,088 | 0.773 | 0.844 | 0.432 | 1.116 | central |
| `260914_0017` | 260914 | 7,000,223 | 0.965 | 0.140 | 0.369 | 1.399 | off-cluster |

The four central entrants span 0.844-0.880 on centrality, so they are matched on where they stand at
trial start and differ on High preference. The 260914 pair are the two most typical central models of
their batch by distance to the cluster centre; the 260723 pair likewise for theirs.

**What self-play already says.** At their finished policies 260914 central leads on score per second
(0.464 against 0.438 for the two 260723 entrants) and chooses High far more often, but is still the
less direct traveller (`path_ratio` 1.164 against about 1.12). Its `mean_score` lead is arithmetic
rather than skill: in self-play each agent wins about half the trials, so mean score follows from
`0.5 * (0.4 + 0.6 * p_high)` and every model in both batches lands within 0.05 of it. Self-play also
under-penalises slowness, because when both agents are equally slow neither loses ground. Competition
punishes it directly, since a faster opponent takes the wall first. That asymmetry is the main reason
to run this.

**Where the code lives.** This notebook is config + visualisation only. Everything that runs or
checks a matchup is in the two shared modules:
- `tournament/tournament_func.py` — assembling a matchup run dir, `run_matchup`, `run_tournament`.
- `tournament/tournament_checks.py` — `preflight`, `smoke_test`, `check_batch`, `rerun_matchups`.

**Read the result with care.** The batches differ in five training settings, which fall into two
packages: a longer credit horizon (gamma 0.99 -> 0.998, lambd 0.95 -> 0.99, `time_horizon` 128 -> 512,
all scaled together) and a different training regime (linear -> constant schedules, `max_steps`
4e6 -> 7e6). A reward gap cannot be pinned on `p_high`, or on any one setting. Note also that 260914
is still improving at 7e6 and was never annealed, so its policies are still fluctuating around their
optimum; if anything this comparison remains tilted towards the settled 260723 models.

**Before running:**
1. Activate the conda env that has `mlagents` (the kernel for this notebook must be that env).
2. Sanity-check CUDA (`nvidia-smi`, `torch.cuda.is_available()`), following the driver notes in `CLAUDE.md`.
3. The build below must be the tournament build with `ScriptedTrialSequence.cs` + the timing keys;
   an older build silently ignores `--trial_seq` and generates random trials.

Actions are **sampled** (not greedy), matching training behaviour, so matchups are not
bit-reproducible; run enough episodes to average over sampling noise. Trial content and timing are
pinned by the predetermined `trial_seq`.

In [5]:
import importlib
import shutil
import sys
from pathlib import Path

# Repo root on sys.path so tournament.* and trainer_and_simulator_functions both resolve.
REPO_ROOT = Path("~/repos/agent_training").expanduser()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reload all three modules. Python caches imports, so editing tournament_func.py does not
# affect a running kernel. The 260730 batch was lost this way: the fixed file was on disk but
# the kernel kept the old functions. If in any doubt, restart the kernel.
import trainer_and_simulator_functions as _tsf
importlib.reload(_tsf)
import tournament.tournament_func as tf
importlib.reload(tf)
import tournament.tournament_checks as tc
importlib.reload(tc)

from tournament.tournament_func import run_tournament

## Config

Everything batch-specific lives in this cell. `MODELS_ROOT` is the parent folder; entrants are its
immediate subfolders (each with `OctagonAgentSocial/checkpoint.pt`). All entrants must live under
one `MODELS_ROOT`, which is why the field is gathered into a combined folder by the next cell.

In [6]:
OCTAGON_DIR = Path("~/Unity/Octagon").expanduser()

# Tournament build (headless Linux). The penalty break is absent, matching how both batches were
# trained, so tournament reward is comparable to training reward for either group.
UNITY_ENV_PATH = Path("~/Unity/Octagon/builds/260909/build_260909_01_tournament_nosteppenalty.x86_64").expanduser()

# Combined field, gathered by the cell below. A mixed-batch tournament cannot point at either
# batch's own results folder, since run_tournament takes entrants from one root. Gathering also
# pins the checkpoints, so training either batch further cannot change a policy mid-tournament.
MODELS_ROOT = Path("~/Unity/Octagon/results/tournament_fields/260914_vs_260723_final").expanduser()

# Each entrant comes from its own batch's live run dir, which holds that batch's FINISHED policy:
# 260914 at ~7.00e6 steps, 260723 at ~4.00e6. Not the 260723_frozen_3e6 snapshot, which is an
# earlier 3e6 policy, and not the stepmatch_4e6 tree.
SOURCE_ROOTS = {
    "260914_0020": OCTAGON_DIR / "results" / "2609" / "260914",
    "260914_0014": OCTAGON_DIR / "results" / "2609" / "260914",
    "260914_0017": OCTAGON_DIR / "results" / "2609" / "260914",
    "260723_0034": OCTAGON_DIR / "results" / "2607" / "260723",
    "260723_0039": OCTAGON_DIR / "results" / "2607" / "260723",
}

# Listed 260914 central -> 260723 central -> off-cluster, so the analysis matrices read with the
# High-preference contrast first and the strategy contrast last.
RUN_IDS = [
    "260914_0020",  # central, p_high 1.000, centrality 0.870
    "260914_0014",  # central, p_high 1.000, centrality 0.880
    "260723_0034",  # central, p_high 0.838, centrality 0.880
    "260723_0039",  # central, p_high 0.773, centrality 0.844
    "260914_0017",  # off-cluster, p_high 0.965, centrality 0.140
]

# Predetermined trial sequence (pins trial content + timing across all matchups).
# Keep it longer than EPISODES, since the build stops when the sequence is exhausted.
TRIAL_SEQ = Path("~/repos/agent_training/trial_sequences/trials_100000_seed17.json").expanduser()

# Where the per-matchup JSON logs land: one folder per batch, so a round-robin is self-contained.
OUT_ROOT = Path("~/Unity/Octagon/simulations/tournaments/260914_vs_260723_final").expanduser()

# MLAgents episodes per matchup. Conversion (measured on 260715 inference): trials ~= episodes - 1.
# Same value as 260715 and 260723, so all batches carry equal weight per matchup.
EPISODES = 350   # ~350 trials/matchup

# Run each distinct pair in both seat orders. P1/P2 is symmetric, so the function defaults
# this off. On here to double the data per pair (2 x ~350 = ~700 trials) and to re-check the
# seat has no effect. On 260715 the mean mirror gap was 0.018, so the assumption held.
# Five entrants with mirrors and self-matchups gives 25 matchups at ~35 min, so ~15 h. Set this
# False to halve it to 15 matchups (~9 h) at the cost of half the trials per pair.
INCLUDE_MIRRORS = True

# Include models playing against themselves, one per entrant, as a ~50% baseline check.
# Needed here: a model's self-matchup reward rate is the baseline its cross-batch reward rate
# has to be read against, since the two groups are not equally matched to begin with.
INCLUDE_SELF_MATCHUPS = True

## Gather and pin the field

Copies each entrant's `configuration.yaml` and `<behaviour>/checkpoint.pt` from its own batch into
`MODELS_ROOT`. Only these two files are read per matchup, so the gathered field stays small. Run
once. It refuses to overwrite an existing field, since a half-replaced one would mix policies
silently.

Pinning matters here because the entrants come from live run dirs. `run_matchup` re-reads each
checkpoint for every matchup over many hours, so resuming training on either batch mid-tournament
would otherwise swap a policy underneath it.

The two batches finish at different step counts by design, so the report below should show ~7.00e6
for 260914 and ~4.00e6 for 260723, tight within each batch.

In [7]:
# take each entrant's finished checkpoint from its own batch and pin them all under one root
if MODELS_ROOT.exists():
    print(f"field already gathered, leaving it alone: {MODELS_ROOT}")
else:
    MODELS_ROOT.mkdir(parents=True)
    for rid in RUN_IDS:
        src = SOURCE_ROOTS[rid] / rid
        ckpt, behaviour = tf.find_checkpoint(src)
        dst = MODELS_ROOT / rid / behaviour
        dst.mkdir(parents=True)
        shutil.copy2(ckpt, dst / "checkpoint.pt")
        shutil.copy2(src / "configuration.yaml", MODELS_ROOT / rid / "configuration.yaml")
        print(f"gathered {rid}  ({behaviour})  from {src.parent.name}")


import torch


def checkpoint_step(ckpt_path):
    """ Return the training step a checkpoint was saved at. Takes the checkpoint path.

        mlagents stores this under a name-mangled key inside global_step, so the lookup is
        guarded: a None means the entrant could not be step-checked. """
    gs = torch.load(ckpt_path, map_location="cpu", weights_only=False).get("global_step")
    if hasattr(gs, "values"):
        gs = next(iter(gs.values()))
    return int(gs.item()) if hasattr(gs, "item") else None


# report the step each entrant sits at. The two batches finish at different lengths by design, so
# what matters is that each batch's own entrants agree, not that the two batches match
steps = {rid: checkpoint_step(tf.find_checkpoint(MODELS_ROOT / rid)[0]) for rid in RUN_IDS}
for rid, s in steps.items():
    print(f"{rid}  step {s:,}" if s is not None else f"{rid}  step UNREADABLE")

for batch, expected in (("260914", 7_000_000), ("260723", 4_000_000)):
    got = [s for r, s in steps.items() if r.startswith(batch) and s is not None]
    if not got:
        continue
    spread = max(got) - min(got)
    flag = "" if spread < 0.01 * expected else "   <-- CHECK, entrants disagree"
    print(f"\n{batch}: {len(got)} entrants, spread {spread:,} steps, "
          f"expected ~{expected/1e6:.2f}e6{flag}")
if None in steps.values():
    print("\nWARNING: at least one step could not be read")

gathered 260914_0020  (OctagonAgentSocial)  from 260914
gathered 260914_0014  (OctagonAgentSocial)  from 260914
gathered 260723_0034  (OctagonAgentSocial)  from 260723
gathered 260723_0039  (OctagonAgentSocial)  from 260723
gathered 260914_0017  (OctagonAgentSocial)  from 260914
260914_0020  step 7,000,371
260914_0014  step 7,000,221
260723_0034  step 4,000,042
260723_0039  step 4,000,088
260914_0017  step 7,000,223

260914: 3 entrants, spread 150 steps, expected ~7.00e6

260723: 2 entrants, spread 46 steps, expected ~4.00e6


In [8]:
# Fail fast on a missing path, a missing checkpoint, or a generated config with no
# environment_parameters/step_penalty. Without that block the build throws on every action
# and the matchup becomes a walkover that still writes a healthy-looking JSON. The check runs
# from here rather than inside tournament_func, so it also catches a stale kernel.
n_matchups = tc.preflight(
    models_root=MODELS_ROOT,
    unity_env_path=UNITY_ENV_PATH,
    trial_seq=TRIAL_SEQ,
    run_ids=RUN_IDS,                        # None => every model under MODELS_ROOT
    include_mirrors=INCLUDE_MIRRORS,
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
    base_port=9100,                     # match the round-robin call below, so preflight
                                        # checks the whole span is free
)

All paths OK, config carries step_penalty. 5 entrants => 25 matchups (with mirrors, incl. self-matchups).


## (Optional) Single-matchup smoke test

Run one matchup with a few episodes first, to confirm the build + checkpoints load and both agents
actually play, before committing to the full round-robin (25 matchups at ~35 min each, so ~15 h).

The pair below is one entrant from each batch, which is the comparison this tournament exists to
make and the first place a cross-batch config mismatch would show up. Both batches share the
`SocialConfig` architecture, so model A's `configuration.yaml` is a valid template for both
behaviours, but this is the matchup that proves it.

In [9]:
# Checks both agents move, the wall triggers are split, and Unity is not throwing on
# step_penalty. A frozen opponent still writes a full-length JSON and exits cleanly, which
# is how two 64-matchup batches were lost.
health = tc.smoke_test(
    model_a_run_dir=MODELS_ROOT / "260914_0020",
    model_b_run_dir=MODELS_ROOT / "260723_0034",
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    trial_seq=TRIAL_SEQ,
    episodes=8,
    base_port=tc.SMOKE_TEST_BASE_PORT,   # 11000, clear of training (5005+20*i, 16 envs
                                         # each) and of the round-robin (9100+)
)

['mlagents-learn', '/home/tom/Unity/Octagon/simulations/tournaments/260914_vs_260723_final/tournament_config.yaml', '--run-id', 'tournament/260914_0020__vs__260723_0034', '--resume', '--inference', '--base-port', '11000', '--env', '/home/tom/Unity/Octagon/builds/260909/build_260909_01_tournament_nosteppenalty.x86_64', '--no-graphics', '--env-args', '--sim_out', '/home/tom/Unity/Octagon/simulations/tournaments/260914_vs_260723_final/_smoke_test', '--sim_eps', '8', '--trial_seq', '/home/tom/repos/agent_training/trial_sequences/trials_100000_seed17.json', '--seed', '17']
smoke-test log: /home/tom/Unity/Octagon/simulations/tournaments/260914_vs_260723_final/_smoke_test/mlagents_stdout.log

trials:           8
unique positions: P1 3429, P2 3175
wall triggers:    P1 5, P2 3
step_penalty exceptions in Unity log: 0

Smoke test passed. Safe to run the full round-robin.


## Full round-robin

Every unordered pair, **including self-matchups**, and (with `INCLUDE_MIRRORS`) each pair in both
seat orders. Five entrants gives 10 pairs, doubled to 20 by mirrors, plus 5 self-matchups = 25.
One JSON per matchup under `OUT_ROOT/<A>__vs__<B>/`. Failures are caught per-matchup (that entry
is `None`) so one bad run does not kill the batch.

The six cross-batch pairs (each `260914` central against each `260723` central, both seat orders)
are the ones that answer the question. The self-matchups give each entrant its own baseline, and
the `260914_0017` pairs carry the separate strategy question over from the single-batch field.

**Self-matchups vs mirrors:**
- **Self-matchups** (a model against a copy of itself) are run, one per entrant, as a ~50% baseline.
- **Mirrors** (the same pairing with the P1/P2 seats swapped) are off in the function by default.
  Which agent takes which seat should be irrelevant: agents are not repositioned at trial start, so
  each agent's position is emergent from its own behaviour, and the only P1/P2 difference is
  administrative (arena setup and logging). They are on here to double the data per pair and to
  re-confirm that the seat has no effect.

In [10]:
logs = run_tournament(
    models_root=MODELS_ROOT,
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    episodes=EPISODES,
    run_ids=RUN_IDS,            # None => all models under MODELS_ROOT
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
    include_mirrors=INCLUDE_MIRRORS,
    trial_seq=TRIAL_SEQ,
    base_port=9100,             # ports 9100 + 20*i -> 9100..10360. Training uses 5005+20*i
                                # per model, so 9100 clears sessions up to ~206 models. Also
                                # above tensorboard (6006) and mDNS (5353).
)

Running 25 matchups over 5 models: ['260914_0020', '260914_0014', '260723_0034', '260723_0039', '260914_0017']

=== [1/25] Matchup: 260914_0020 (P1) vs 260914_0020 (P2) ===
['mlagents-learn', '/home/tom/Unity/Octagon/simulations/tournaments/260914_vs_260723_final/tournament_config.yaml', '--run-id', 'tournament/260914_0020__vs__260914_0020', '--resume', '--inference', '--base-port', '9100', '--env', '/home/tom/Unity/Octagon/builds/260909/build_260909_01_tournament_nosteppenalty.x86_64', '--no-graphics', '--env-args', '--sim_out', '/home/tom/Unity/Octagon/simulations/tournaments/260914_vs_260723_final/260914_0020__vs__260914_0020', '--sim_eps', '350', '--trial_seq', '/home/tom/repos/agent_training/trial_sequences/trials_100000_seed17.json', '--seed', '17']
=== Finished: 260914_0020__vs__260914_0020 -> /home/tom/Unity/Octagon/simulations/tournaments/260914_vs_260723_final/260914_0020__vs__260914_0020/mlagents_stdout.log ===

=== [2/25] Matchup: 260914_0020 (P1) vs 260914_0014 (P2) ===
['

In [11]:
# Which matchups ran correctly? A matchup can exit cleanly with a full-length JSON and still
# be void: before the 260729 fix a missing environment_parameters block froze P2 in all 64
# matchups. Only suspect matchups are printed.
suspect = tc.check_batch(
    out_root=OUT_ROOT,
    tournament_ids=logs.keys(),
    expected_trials=EPISODES,
    octagon_dir=OCTAGON_DIR,
    logs=logs,                              # so matchups that raised are reported too
)

25/25 matchups ran without error

Validity check (both agents move, both win some trials):

25/25 matchups look valid.


## Rerun specific matchups

For matchups that failed or came out short above, without re-running the whole batch. List the
ordered `(P1, P2)` pairs in `RERUN` — the same ids that make up the matchup dir name
`<P1>__vs__<P2>`.

Everything else (build, checkpoints, trial sequence, sampled actions) is identical to the
round-robin, so the reruns are drop-in replacements for the originals.

**On stalled matchups.** In `260715` the `0002` self-matchup stalled rather than crashed:
`No episode was completed since last summary` in its stdout, and 68 `trial start` events in
~7200 s against ~350 in ~2200 s for a healthy matchup. Both copies wandered without triggering a
wall. Actions are sampled, so changing `seed` re-rolls the trajectories and is the lever most
likely to change the outcome. Raising `timeout_s` alone tends to buy a longer stall. Check the
trial count in the health line, not just the absence of an exception.

`260914_0017` is the entrant to watch. It is the slowest model of its batch to reach a wall
(median 2.85 s against 2.09 s for the central cluster) and takes the longest route, so its
matchups have the least headroom against `timeout_s`.

In [12]:
# Ordered (P1, P2) run_id pairs to rerun. Same id twice => self-matchup.
RERUN = [
    # ("260914_0020", "260723_0034"),
]

rerun_logs = tc.rerun_matchups(
    pairs=RERUN,
    models_root=MODELS_ROOT,
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,          # same dir as the round-robin, so reruns land beside siblings
    episodes=EPISODES,          # trials ~= episodes - 1
    trial_seq=TRIAL_SEQ,
    seed=17,                    # change to re-roll the sampled trajectories
    timeout_s=5000,             # wall-clock cap per matchup (healthy 350-ep run: ~2200 s)
    base_port=10500,            # clear of the round-robin's 9100..10360
    archive_existing=True,      # move the old matchup dir to OUT_ROOT/../superseded/ first
)

no pairs to rerun


## Resume an interrupted batch

`run_tournament` always starts at matchup 1, so re-running it after an interruption redoes work
that is already done. Worse, `run_eval` deletes `DONE.txt` before launching, so a restart that dies
early strips the marker off a matchup that had finished. That happened on 2026-09-10: the batch was
interrupted at matchup 55, restarted, and overwrote the first matchup's `DONE.txt` before dying.
The 350-trial JSON survived and the fragment is in `tournaments/superseded/`.

Use this cell instead. `outstanding_matchups` reads what is on disk and returns only the pairs still
needed, so it is safe to run repeatedly until it reports nothing outstanding.

In [13]:
# Pairs still needed: missing dir, no JSON, no DONE.txt, or failing the health check.
todo = tc.outstanding_matchups(
    out_root=OUT_ROOT,
    run_ids=RUN_IDS,
    expected_trials=EPISODES,
    include_mirrors=INCLUDE_MIRRORS,
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
)

rerun_logs = tc.rerun_matchups(
    pairs=todo,
    models_root=MODELS_ROOT,        # still the gathered, pinned field
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    episodes=EPISODES,
    trial_seq=TRIAL_SEQ,
    seed=17,
    timeout_s=5000,
    base_port=10500,                # clear of the round-robin's 9100..10360
    archive_existing=True,          # partial dirs move to ../superseded/ before the rerun
)

25/25 matchups complete, 0 outstanding
no pairs to rerun


In [14]:
# Confirm the batch is complete. Expect "0 outstanding" and every matchup valid.
tc.outstanding_matchups(
    out_root=OUT_ROOT,
    run_ids=RUN_IDS,
    expected_trials=EPISODES,
    include_mirrors=INCLUDE_MIRRORS,
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
)

ids = ["__vs__".join(p) for p in
       tc.expected_matchups(RUN_IDS, INCLUDE_MIRRORS, INCLUDE_SELF_MATCHUPS)]
suspect = tc.check_batch(OUT_ROOT, ids, expected_trials=EPISODES, octagon_dir=OCTAGON_DIR)

25/25 matchups complete, 0 outstanding

Validity check (both agents move, both win some trials):

25/25 matchups look valid.
